# 07.07 - Custom Dataset and DataLoader

**Daily output:** `ImageDataset` from a CSV with `image_path,label`, plus printed batch shapes and labels.

Today turns image files into model-ready batches: CSV metadata, label mapping, custom `Dataset`, transforms, `DataLoader`, train/validation behavior, imbalance checks, and visual sanity checks.

**Notebook type:** Solution notebook with full working code.


## Dataset/DataLoader Mental Model

A `Dataset` answers two questions: how many examples exist, and how do I load one example? A `DataLoader` handles batching, shuffling, workers, and stacking examples into tensors.

For image classification, a CSV usually has `image_path` and `label`. The model needs integer class IDs, so create a stable mapping like `{'cat': 0, 'dog': 1}`.


In [ ]:
from pathlib import Path
import csv
import random

import numpy as np
from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed. Visualization examples will be skipped.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("_day07_image_data")
IMG_DIR = ROOT / "images"
CSV_PATH = ROOT / "labels.csv"
IMG_DIR.mkdir(parents=True, exist_ok=True)


## Create a Tiny Image Dataset

This notebook generates a synthetic image dataset so it runs without downloads. In a contest, replace this with the provided CSV and image folder.


In [ ]:
def make_colored_image(label, size=(80, 80), seed=0):
    rng = np.random.default_rng(seed)
    base = {
        "red": np.array([220, 40, 40], dtype=np.uint8),
        "green": np.array([40, 180, 80], dtype=np.uint8),
        "blue": np.array([50, 90, 220], dtype=np.uint8),
    }
    arr = np.zeros((size[1], size[0], 3), dtype=np.uint8)
    arr[...] = base[label]
    noise = rng.normal(0, 18, size=arr.shape).astype(np.int16)
    arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr, mode="RGB")
    ImageDraw.Draw(img).rectangle([10, 10, 30, 30], outline=(255, 255, 255), width=2)
    return img

counts = {"red": 16, "green": 10, "blue": 6}
rows = []
for label, count in counts.items():
    for i in range(count):
        path = IMG_DIR / f"{label}_{i:02d}.png"
        make_colored_image(label, seed=1000 + i).save(path)
        rows.append({"image_path": str(path), "label": label})
random.shuffle(rows)

with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
    writer.writeheader()
    writer.writerows(rows)

print("wrote", CSV_PATH)
print("rows:", len(rows))
print(rows[:3])


## Read CSV and Build Label Mapping

The exact integer IDs do not matter as much as consistency. Save `label_to_id` and `id_to_label` with your experiment notes or checkpoint.


In [ ]:
def read_label_csv(csv_path):
    with Path(csv_path).open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

samples = read_label_csv(CSV_PATH)
unique_labels = sorted({row["label"] for row in samples})
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
print("first row:", samples[0])
print("label_to_id:", label_to_id)


## Minimal Transforms

Writing these once makes the image contract memorable: PIL RGB image in, `[C, H, W]` float tensor out.


In [ ]:
class Resize:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        return img.resize(self.size, resample=Image.BILINEAR)

class ToTensor:
    def __call__(self, img):
        arr = np.array(img.convert("RGB"), dtype=np.float32) / 255.0
        return torch.from_numpy(np.transpose(arr, (2, 0, 1)))

class Normalize:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean, dtype=torch.float32)[:, None, None]
        self.std = torch.tensor(std, dtype=torch.float32)[:, None, None]

    def __call__(self, tensor):
        return (tensor - self.mean) / self.std

class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, x):
        for transform in self.transforms:
            x = transform(x)
        return x

train_transform = Compose([Resize((64, 64)), ToTensor(), Normalize([0.5] * 3, [0.5] * 3)])
val_transform = Compose([Resize((64, 64)), ToTensor(), Normalize([0.5] * 3, [0.5] * 3)])


## Build `ImageDataset`

Keep the Dataset focused: load one row, open image, transform it, map the label, and return useful metadata for debugging.


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, rows, label_to_id, transform=None, path_col="image_path", label_col="label"):
        self.rows = list(rows)
        self.label_to_id = dict(label_to_id)
        self.transform = transform
        self.path_col = path_col
        self.label_col = label_col

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        path = Path(row[self.path_col])
        label_name = row[self.label_col]
        if label_name not in self.label_to_id:
            raise KeyError(f"Unknown label {label_name!r}")
        label_id = self.label_to_id[label_name]

        with Image.open(path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)

        return {
            "image": img,
            "label": torch.tensor(label_id, dtype=torch.long),
            "label_name": label_name,
            "path": str(path),
        }

full_ds = ImageDataset(samples, label_to_id, transform=train_transform)
item = full_ds[0]
print(item.keys())
print("image:", item["image"].shape, item["image"].dtype, item["image"].min().item(), item["image"].max().item())
print("label:", item["label"], item["label_name"])


## DataLoader Batches

The default collate function stacks tensors and keeps strings as lists/tuples. For images, your expected batch shape is `[B, C, H, W]`.


In [ ]:
loader = DataLoader(full_ds, batch_size=8, shuffle=True, num_workers=0)
batch = next(iter(loader))
print("keys:", batch.keys())
print("images:", batch["image"].shape, batch["image"].dtype)
print("labels:", batch["label"].shape, batch["label"].dtype, batch["label"].tolist())
print("label names:", batch["label_name"])
print("paths first 2:", batch["path"][:2])


## Visual Batch Sanity Check

Always visualize a batch before training. It catches wrong labels, wrong colors, broken normalization, and accidental path mistakes.


In [ ]:
def unnormalize_for_display(tensor, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
    mean = torch.tensor(mean)[:, None, None]
    std = torch.tensor(std)[:, None, None]
    x = (tensor.cpu() * std + mean).clamp(0, 1)
    return x.permute(1, 2, 0).numpy()

if plt is not None:
    batch = next(iter(loader))
    fig, axes = plt.subplots(2, 4, figsize=(8, 4))
    for ax, image, label_id in zip(axes.ravel(), batch["image"][:8], batch["label"][:8]):
        ax.imshow(unnormalize_for_display(image))
        ax.set_title(id_to_label[int(label_id)])
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib to visualize a batch.")


## Train/Validation Split

For a tiny demo, `random_split` is fine. For real competitions, use a stratified split so every class appears in both train and validation.


In [ ]:
generator = torch.Generator().manual_seed(SEED)
train_size = int(0.75 * len(full_ds))
val_size = len(full_ds) - train_size
train_subset, val_subset = random_split(full_ds, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=8, shuffle=False, num_workers=0)

print("train examples:", len(train_subset))
print("val examples:", len(val_subset))
print("train batch:", next(iter(train_loader))["image"].shape)
print("val batch:", next(iter(val_loader))["image"].shape)


## Imbalance Check

Count classes before training. If classes are imbalanced, consider class-weighted loss, a sampler, more data, or metric-aware thresholding.


In [ ]:
def count_labels(rows):
    counts = {}
    for row in rows:
        counts[row["label"]] = counts.get(row["label"], 0) + 1
    return counts

print("counts:", count_labels(samples))

labels_as_ids = [label_to_id[row["label"]] for row in samples]
class_counts = torch.bincount(torch.tensor(labels_as_ids))
class_weights = 1.0 / class_counts.float()
sample_weights = [class_weights[label_id].item() for label_id in labels_as_ids]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
balanced_loader = DataLoader(full_ds, batch_size=12, sampler=sampler, num_workers=0)
balanced_batch = next(iter(balanced_loader))
print("sampled labels:", [id_to_label[int(i)] for i in balanced_batch["label"]])


## Robust Dataset Variant

During development, failing fast is best. For long training runs, you may choose a placeholder image and keep an error flag.


In [ ]:
class RobustImageDataset(ImageDataset):
    def __init__(self, *args, placeholder_size=(64, 64), strict=True, **kwargs):
        super().__init__(*args, **kwargs)
        self.placeholder_size = placeholder_size
        self.strict = strict

    def __getitem__(self, idx):
        try:
            return super().__getitem__(idx)
        except Exception as exc:
            if self.strict:
                raise
            row = self.rows[idx]
            label_name = row[self.label_col]
            label_id = self.label_to_id[label_name]
            placeholder = Image.new("RGB", self.placeholder_size, (0, 0, 0))
            image = self.transform(placeholder) if self.transform else placeholder
            return {
                "image": image,
                "label": torch.tensor(label_id, dtype=torch.long),
                "label_name": label_name,
                "path": row[self.path_col],
                "load_error": str(exc),
            }

broken_rows = samples[:3] + [{"image_path": str(ROOT / "missing.png"), "label": "red"}]
robust_ds = RobustImageDataset(broken_rows, label_to_id, transform=val_transform, strict=False)
for i in range(len(robust_ds)):
    item = robust_ds[i]
    print(i, item["image"].shape, item["label_name"], item.get("load_error", "ok"))


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 07 tests passed`.


In [ ]:
def run_day07_tests():
    required_names = [
        "read_label_csv",
        "Resize",
        "ToTensor",
        "Normalize",
        "Compose",
        "ImageDataset",
        "count_labels",
        "RobustImageDataset",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    test_samples = read_label_csv(CSV_PATH)
    assert len(test_samples) == len(samples)
    assert {"image_path", "label"}.issubset(test_samples[0].keys())
    assert set(label_to_id.keys()) == {"blue", "green", "red"}

    test_ds = ImageDataset(test_samples, label_to_id, transform=val_transform)
    item = test_ds[0]
    assert set(["image", "label", "label_name", "path"]).issubset(item.keys())
    assert item["image"].shape == (3, 64, 64), f"Expected CHW image, got {item['image'].shape}"
    assert item["image"].dtype == torch.float32
    assert item["label"].dtype == torch.long
    assert item["label_name"] in label_to_id
    assert Path(item["path"]).exists()

    test_loader = DataLoader(test_ds, batch_size=5, shuffle=False, num_workers=0)
    batch = next(iter(test_loader))
    assert batch["image"].shape == (5, 3, 64, 64)
    assert batch["label"].shape == (5,)
    assert batch["label"].dtype == torch.long

    counts = count_labels(test_samples)
    assert counts == {"red": 16, "green": 10, "blue": 6}, counts
    assert sum(counts.values()) == len(test_samples)

    broken_rows = test_samples[:2] + [{"image_path": str(ROOT / "missing.png"), "label": "red"}]
    robust_ds = RobustImageDataset(broken_rows, label_to_id, transform=val_transform, strict=False)
    bad_item = robust_ds[2]
    assert "load_error" in bad_item
    assert bad_item["image"].shape == (3, 64, 64)
    assert bad_item["label"].dtype == torch.long

    print("Day 07 tests passed")

run_day07_tests()


## Day 07 Checklist

Check CSV columns, image paths, label mapping, one dataset item, one DataLoader batch, train loader shuffle, validation loader no shuffle, transform differences, class counts, and a visual batch.
